# Pipeline stage visualization

Static replacement for the former `debug_app.py` Tk GUI. Runs the pipeline **once**
on a single page, then for every stage renders:

- one **original page** image, then
- for each overlay category of that stage, a pair: the category's geometry drawn
  alone on white (**isolated**), and the original page with that geometry drawn on
  top (**overlay**).

So a single-overlay stage produces 3 images; a stage with *N* overlays produces
`1 + 2N`. `clustering` and `color_separation` expand **every** category / bucket,
which can be 100+ images on a dense page.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "pipeline_stage_visualization.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rastervec.logging_setup import configure_logging
from rastervec.native_text import render_native
from rastervec.OCR.fast_detect import render_fast
from rastervec.OCR.Paddle_OCR.render_ocr import render_ocr_results
from rastervec.OCR.radon import render_radon
from rastervec.pipelines._steps import render_drawing, render_regroup
from rastervec.pipelines.current import STEP_NAMES, run_pipeline
from rastervec.renderer.notebook import RenderResult, page_setup, visualize
from rastervec.Vector.vector import render_vectors
from rastervec.Vector_Classification.classification import (
    render_clustering_steps,
    render_layer_color_buckets,
    render_layers,
    render_text_candidates,
)

configure_logging()

## Parameters

`VARIANT` picks one of the `current` pipeline variants
(`Evaluation/Evaluate/variants.py`: `current`, `current_nofast`) — it sets
`enable_fast` exactly as the benchmark runs it. The `legacy` variant is not
runnable here (it's `archive/raster_parser`, not this pipeline).

In [ ]:
from rastervec.Evaluation.Evaluate.variants import VARIANTS, resolve_variant

PDF_PATH = next(iter(sorted((PROJECT_ROOT / "references").glob("*.pdf"))), None)
PAGE_INDEX = 3
ZOOM = 2.0

# Which pipeline variant to visualize. Only "current"-engine variants run here
# (legacy is archive/raster_parser, not this pipeline): "current", "current_nofast".
# The variant sets enable_fast below.
VARIANT = "current"

_variant = resolve_variant(VARIANT)
assert _variant.engine == "current", (
    f"{VARIANT!r} is engine={_variant.engine!r}; only 'current' variants run in this notebook. "
    f"current variants: {[n for n, v in VARIANTS.items() if v.engine == 'current']}"
)
ENABLE_FAST = _variant.enable_fast

assert PDF_PATH is not None, "no PDF under references/ -- set PDF_PATH by hand"
print("PDF:", PDF_PATH, "| page", PAGE_INDEX)
print(f"variant: {_variant.name}  (enable_fast={_variant.enable_fast})")

## Run the pipeline

In [ ]:
res = run_pipeline(str(PDF_PATH), PAGE_INDEX, enable_fast=_variant.enable_fast, verbose=True)
outputs = res.step_outputs or {}
page = res.page
assert page is not None, "reader step failed"

for name, o in outputs.items():
    print(f"{'ok ' if o.status == 'ok' else 'ERR'}  {name:10}  {o.error or ''}")


## Setup

In [ ]:
# Every render_<stage_name> function lives next to its stage's own code
# (rastervec.native_text, rastervec.Vector.vector, ...) and reads `res`
# directly; this is the one bit of shared setup they all need: the
# rasterized original page at ZOOM, plus the rotation-baked display matrix.
ORIGINAL, MATRIX = page_setup(res, ZOOM)

## 1. Reader

In [ ]:
visualize(
    "read",
    RenderResult(note=(
        f"mediabox={page.meta.mediabox}  rotation={page.meta.rotation}  "
        f"size={page.meta.width:.0f}x{page.meta.height:.0f}"
    )),
    step_outputs=outputs, original=ORIGINAL, matrix=MATRIX,
)

## 2. Native Text

In [ ]:
visualize("native", render_native(res, zoom=ZOOM), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 3. Vector Extraction

In [ ]:
visualize("vectors", render_vectors(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 4. Layer + Color Separation  (inside `classify`)


In [ ]:
visualize("classify", render_layers(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

In [ ]:
visualize("classify", render_layer_color_buckets(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 5. Clustering  (the 12-step chain, per bucket)


In [ ]:
visualize("classify", render_clustering_steps(res, MATRIX, ORIGINAL), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 6. Text Candidates  (`classify`)

In [ ]:
visualize("classify", render_text_candidates(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 7. FAST: Text Detect

In [ ]:
fr = res.fast_result
if fr is not None:
    passed_set = res.fast_passed or []
    for c in passed_set + (res.fast_dropped or []):
        print(f"  score {fr.scores.get(id(c), 0.0) * 100:5.1f}%  {'PASS' if c in passed_set else 'drop'}  {len(c)} path(s)")
visualize("fast", render_fast(res, enable_fast=ENABLE_FAST), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 8. Spatial Regroup


In [ ]:
visualize("regroup", render_regroup(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 9. Segment (Radon)

In [ ]:
visualize("segment", render_radon(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 10. PaddleOCR

In [ ]:
cor = res.cluster_ocr_results or []
for r in cor:
    if r.resolved.text.strip():
        print(f"  {r.resolved.text!r:42}  conf={r.resolved.confidence:.2f}  rot={r.resolved.rotation_used:>3}  {r.ocr_seconds:.2f}s")
visualize("ocr", render_ocr_results(res, zoom=ZOOM), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## 11. Drawing Vectors

In [ ]:
visualize("drawing", render_drawing(res, zoom=ZOOM), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX)

## Reading the results

- Everything a step **drops** (`classify` side categories, `fast` dropped, `ocr`
  blank readings) is folded into `drawing_vectors` -- the pipeline's "this is
  drawing content, not text" verdict.
- `ocr` treats a **blank** reading as a failure; a non-blank reading survives into
  the final output.
- The pipeline always runs all 9 steps (including PaddleOCR); the first run
  downloads the PaddleOCR weights. `enable_fast` comes from the chosen `VARIANT`
  (`current` = on, `current_nofast` = pass-through).